# Data Integrity

In [ ]:
# General TODOs:

# TODO: Generalize the data processing pipeline
# TODO: Add plots where needed (timestamps, nans, spreads, etc.)
# TODO: Redo NaNs analysos for 1st level only

# General NOTEs:

# NOTE: All the analysis results are stored in data/interim/data-integrity/, no
# need to run them again

# Setup

## Imports

In [ ]:
import hashlib
import warnings
from collections import Counter, defaultdict
from datetime import time
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import (
    FuncFormatter,
    MultipleLocator,
    PercentFormatter,
)

from thesis_project import (
    collectors,
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "data-integrity"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

## Utilities

In [ ]:
from thesis_project.utils.dates import filename_to_date


def count_unique_lobs(root: Path) -> int:
    """
    Count the number of unique LOBs in a directory.
    """
    files = [p.name for p in root.rglob("*.parquet")]
    unique_files = set(files)
    return len(unique_files)


def dups_by_name(root: Path) -> List[str]:
    """
    Check for duplicate file names in a directory.
    """
    files = [p.name for p in root.rglob("*.parquet")]
    dups = [name for name, c in Counter(files).items() if c > 1]

    return dups


def dups_by_hash(root: Path) -> List[List[Path]]:
    """
    Check for duplicate files by grouping them by size and by hash.
    """

    def file_hash(path: Path, chunk_size: int = 1 << 20) -> str:
        h = hashlib.md5()
        with open(path, "rb") as f:
            while chunk := f.read(chunk_size):
                h.update(chunk)
        return h.hexdigest()

    size_map = defaultdict(list)
    for p in root.rglob("*.parquet"):
        size_map[p.stat().st_size].append(p)
    dups = []

    for paths in size_map.values():
        if len(paths) < 2:
            continue

        hash_map = defaultdict(list)
        for p in paths:
            hash_map[file_hash(p)].append(p)

        for group in hash_map.values():
            if len(group) > 1:
                dups.append(group)

    return dups


def time_to_seconds(t: time, cutoff: time = time(1, 0, 0)) -> int:
    s = t.hour * 3600 + t.minute * 60 + t.second
    cutoff_s = cutoff.hour * 3600 + cutoff.minute * 60 + cutoff.second
    if s < cutoff_s:
        s += 24 * 3600
    return s


def add_issue(filename: str, lob_type: str, id: str, description: str):
    """
    Add a reported issue to the issues log CSV file.
    If the file doesn't exist, it will be created.

    Args:
        filename: The name of the file with the issue.
        lob_type: The type of LOB (e.g., "futures", "ctd").
        id: A unique identifier for the issue (e.g., "spread_consistency").
        description: A detailed description of the issue.
    """
    file_path = OUT_PRO_DIR / "lob_issues.csv"
    if file_path.exists():
        issues = pd.read_csv(file_path, index_col="Date", parse_dates=["Date"])
    else:
        issues = pd.DataFrame(columns=["Filename", "LOB type", "Issue ID", "Issue description"])
        issues.index.name = "Date"

    date = filename_to_date(filename)
    date = pd.to_datetime(date) if date is not None else pd.NaT
    issue = pd.DataFrame(
        {
            "Filename": [filename],
            "LOB type": [lob_type],
            "Issue ID": [id],
            "Issue description": [description],
        },
        index=[date],
    )
    issue.index.name = "Date"

    if issues.empty:
        issues = issue
    else:
        issues = pd.concat([issues, issue])
    issues = issues.sort_index()

    issues = issues.drop_duplicates(subset=["Filename", "LOB type", "Issue ID"], keep="first")

    issues.to_csv(file_path)

### Plotting

In [ ]:
def plot_timestamp_distribution(
    df: pd.DataFrame,
    timestamp_col: str,
    bin_start_hh_mm: str = "14:00",
    bin_width_seconds: int = 450,
    major_tick_minutes: int = 30,
):
    """
    Plot the distribution of timestamps in a DataFrame as a histogram.
    Converts time objects to minutes, bins them, and displays frequency distribution
    as percentages. X-axis shows time in HH:MM format with customizable binning.

    Args:
        df: DataFrame containing the timestamp column.
        timestamp_col: Name of the column with timestamps (format HH:MM:SS).
        bin_start_hh_mm: Starting point for bins in HH:MM format (default "14:00").
        bin_width_seconds: Width of each bin in seconds (default 450s = 7.5min).
        major_tick_minutes: Interval for major ticks on x-axis in minutes (default 30min).
    """
    # Parse timestamps and convert to minutes
    ts_parsed = pd.to_datetime(
        df[timestamp_col].astype(str), format="%H:%M:%S", errors="coerce"
    ).dropna()

    # Convert to total minutes from midnight
    minutes_from_midnight = (
        ts_parsed.dt.hour * 60 + ts_parsed.dt.minute + ts_parsed.dt.second / 60.0
    )
    next_day_mask = minutes_from_midnight < 60
    minutes_from_midnight[next_day_mask] += 24 * 60

    # Calculate bin edges
    start_h, start_m = map(int, bin_start_hh_mm.split(":"))
    start_min = start_h * 60 + start_m
    bin_width_min = bin_width_seconds / 60.0
    end_min = np.ceil(minutes_from_midnight.max() / bin_width_min) * bin_width_min
    bin_edges = np.arange(start_min, end_min + bin_width_min, bin_width_min)

    # Create formatters and figure
    def min_to_hh_mm(minutes: int, pos: int) -> str:
        """Convert minutes from midnight to HH:MM format."""
        hours = int(minutes // 60)
        mins = int(minutes % 60)
        if minutes >= 24 * 60:
            out = f"{hours-24:02d}:{mins:02d}"
        else:
            out = f"{hours:02d}:{mins:02d}"
        return out

    x_formatter = FuncFormatter(min_to_hh_mm)
    y_formatter = PercentFormatter(1)

    # Create histogram
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(
        minutes_from_midnight,
        bins=bin_edges,  # type: ignore
        weights=np.full(len(minutes_from_midnight), 1 / len(minutes_from_midnight)),
    )

    # Configure axes
    bin_width_title = f"{bin_width_seconds // 60:02d}:{bin_width_seconds % 60:02d}"
    ax.set_title(
        f"Distribution of {timestamp_col.lower()}\n"
        f"{len(minutes_from_midnight)} samples, {minutes_from_midnight.nunique()} unique values\n"
        f"{bin_width_title} bins from {bin_start_hh_mm}",
        # fontsize=12,
        # fontweight="bold",
    )
    ax.set_xlabel(
        "Time (HH:MM)",
        # fontsize=11
    )
    ax.set_ylabel(
        "Frequency (%)",
        # fontsize=11
    )

    # Format and grid
    ax.xaxis.set_major_formatter(x_formatter)
    ax.xaxis.set_major_locator(MultipleLocator(major_tick_minutes))
    ax.xaxis.set_minor_locator(MultipleLocator(bin_width_seconds / 60))
    ax.yaxis.set_major_formatter(y_formatter)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.3)

# Main

## Common

In [ ]:
# Define collectors functions
run_collectors_single = collectors.run_collectors_single
run_collectors_double = collectors.run_collectors_double

# Define specific collectors
extractors = {
    "Empty LOBs": collectors.collect_empty_lobs,
    "NaNs": collectors.collect_nans,
    "Timestamps": collectors.collect_timestamps,
    "FUT specific": collectors.collect_fut_integrity,
    "CTD specific": collectors.collect_ctd_integrity,
    "Spread sign": collectors.collect_spread_sign,
    "Volume sign": collectors.collect_volume_sign,
    "Price order": collectors.collect_bid_ask_order,
}

## Duplicates

Quick check for duplicates in the datasets.

In [ ]:
# Count unique files
fut_unique = count_unique_lobs(LOB_FUT_DIR)
ctd_unique = count_unique_lobs(LOB_CTD_DIR)

# Check for duplicates by hash
fut_dups_name = dups_by_name(LOB_FUT_DIR)
ctd_dups_name = dups_by_name(LOB_CTD_DIR)

# Check for duplicates by hash
fut_dups_hash = dups_by_hash(LOB_FUT_DIR)
ctd_dups_hash = dups_by_hash(LOB_CTD_DIR)

fut_summary = pd.DataFrame(
    {
        "Unique LOBs": fut_unique,
        "Number of duplicates by name": len(fut_dups_name),
        "Number of duplicates by hash": len(fut_dups_hash),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "Unique LOBs": ctd_unique,
        "Number of duplicates by name": len(ctd_dups_name),
        "Number of duplicates by hash": len(ctd_dups_hash),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} duplicates summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}
display(summary)
display(summary.attrs)

# filename = "duplicates_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## FUT specific issues

Analyze future LOBs after market closing 

In [ ]:
key = "FUT specific"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_specific_checks.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures post 19:00 integrity checks",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}

display(fut_df)
display(fut_df.attrs)

# filename = "specific_checks.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")

In [ ]:
non_empty = fut_df["Rows"] > 0
fut_df_ne = fut_df[non_empty]
n_lobs = len(fut_df)
n_lobs_ne = len(fut_df_ne)
ts = fut_df_ne["Rows"].sum()
ts_e_any = fut_df["Rows any NaNs"].sum()
ts_e = fut_df["Rows all NaNs"].sum()
n_cells = ts * 40
n_empty_cells = fut_df_ne["Total NaNs"].sum()
full_lvl = fut_df_ne["Cols all NaNs"] <= 36

summary = pd.DataFrame(
    {
        "Total LOBs": n_lobs,
        "Non-empty LOBs": n_lobs_ne,
        "Non-empty LOBs (%)": n_lobs_ne / n_lobs * 100,
        "Total timestamps": ts,
        "Partially empty timestamps": ts_e_any,
        "Partially empty timestamps (%)": ts_e_any / ts * 100,
        "Fully empty timestamps": ts_e,
        "Fully empty timestamps (%)": ts_e / ts * 100,
        "Total cells": n_cells,
        "Total empty cells": n_empty_cells,
        "Total empty cells (%)": n_empty_cells / n_cells * 100,
        "LOBs with full 1st level (at least somewhere)": full_lvl.sum(),
    },
    index=["FUT"],
)
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} futures post 19:00 integrity checks summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}
display(summary.T.style.format("{:.1f}"))
display(summary.attrs)

# filename = "fut_specific_checks_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## CTD specific issues

Analyze CTD LOBs first row

In [ ]:
key = "CTD specific"
func = extractors[key]
extractor = {key: func}

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_specific_checks.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd first timestamps integrity checks",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}

display(ctd_df)
display(ctd_df.attrs)

# filename = "specific_checks.parquet"
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
summary = pd.DataFrame(
    {
        "Total LOBs": len(ctd_df),
        "First timestamp empty": ctd_df["First timestamp empty"].sum(),
        "First timestamp empty (%)": ctd_df["First timestamp empty"].mean() * 100,
    },
    index=["CTD"],
)
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} ctd first timestamps integrity checks summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}
display(summary)
display(summary.attrs)

# filename = "ctd_specific_checks_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## Empty LOBs
 Analyze empty LOBs in the dataset.

In [ ]:
key = "Empty LOBs"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.FUT_MRKT_OPEN,
    end_time=config.FUT_MRKT_CLOSE,
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_empty_lobs.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures empty LOBs",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.FUT_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.FUT_MRKT_CLOSE.strftime("%H:%M"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_empty_lobs.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd empty LOBs",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.CTD_MRKT_OPEN.strftime("%H:%M:%S"),
    "End time": config.CTD_MRKT_CLOSE.strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "empty_lobs.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
fut_summary = pd.DataFrame(
    {
        "File processed": len(fut_df),
        "Empty LOBs": fut_df["Empty"].sum(),
        "Empty LOBs (%)": fut_df["Empty"].mean() * 100,
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "File processed": len(ctd_df),
        "Empty LOBs": ctd_df["Empty"].sum(),
        "Empty LOBs (%)": ctd_df["Empty"].mean() * 100,
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} empty LOBs summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": config.FUT_MRKT_OPEN.strftime("%H:%M:%S"),
    "Fut end time": config.FUT_MRKT_CLOSE.strftime("%H:%M:%S"),
    "CTD start time": config.CTD_MRKT_OPEN.strftime("%H:%M:%S"),
    "CTD end time": config.CTD_MRKT_CLOSE.strftime("%H:%M:%S"),
}
display(summary)
display(summary.attrs)

# filename = "empty_lobs_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## NaNs Analysis
Analyze NaNs in the dataset.

In [ ]:
key = "NaNs"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.FUT_MRKT_OPEN,
    end_time=config.FUT_MRKT_CLOSE,
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_nans.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures NaNs analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.FUT_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.FUT_MRKT_CLOSE.strftime("%H:%M"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_nans.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd NaNs analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.CTD_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.CTD_MRKT_CLOSE.strftime("%H:%M"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "nans.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
# Show top days with all NaNs on the rows
print("FUT:")
display(fut_df.sort_values(by="Rows all NaNs", ascending=False).head())
print("\nCTD:")
display(ctd_df.sort_values(by="Rows all NaNs", ascending=False).head())

In [ ]:
# Show top days with any NaNs on the rows
print("FUT:")
display(fut_df.sort_values(by="Rows any NaNs", ascending=False).head())
print("\nCTD:")
display(ctd_df.sort_values(by="Rows any NaNs", ascending=False).head())

In [ ]:
# Show top days with total NaNs
print("FUT:")
display(fut_df.sort_values(by="Total NaNs", ascending=False).head())
print("\nCTD:")
display(ctd_df.sort_values(by="Total NaNs", ascending=False).head())

## Spread consistency

Check for spread consistency in the dataset.

In [ ]:
key = "Spread sign"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_spread_sign.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures spread consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_spread_sign.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd spread consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "spread_sign.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
# fut_df = pd.read_parquet(OUT_INT_DIR / "fut_spread_sign.parquet")
# ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_spread_sign.parquet")

fut_summary = pd.DataFrame(
    {
        "File processed": len(fut_df),
        "LOB-wise spread sign violations": (~fut_df["Overall spread check"]).sum(),
        "LOB-wise spread sign violations (%)": (~fut_df["Overall spread check"]).mean() * 100,
        "Global consistency": fut_df["Overall spread check"].all(),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "File processed": len(ctd_df),
        "LOB-wise spread sign violations": (~ctd_df["Overall spread check"]).sum(),
        "LOB-wise spread sign violations (%)": (~ctd_df["Overall spread check"]).mean() * 100,
        "Global consistency": ctd_df["Overall spread check"].all(),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} spread consistency summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "Fut end time": time(23, 59, 59).strftime("%H:%M:%S"),
    "CTD start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "CTD end time": time(23, 59, 59).strftime("%H:%M:%S"),
}
display(summary)
display(summary.attrs)

# filename = "spread_sign_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

In [ ]:
levels = range(1, config.LEVELS + 1)
bid_cols = [f"L{i}-BidPrice" for i in levels]
ask_cols = [f"L{i}-AskPrice" for i in levels]

idx = fut_df.loc[~fut_df["Overall spread check"]].index

display(fut_df.loc[idx])

for id in idx:
    filename = fut_df.loc[id, "Filename"]

    date = filename_to_date(filename)  # type: ignore
    year = str(date.year)  # type: ignore
    month = str(date.month).zfill(2)  # type: ignore

    df = pd.read_parquet(LOB_FUT_DIR / year / month / filename)  # type: ignore

    bids = df[bid_cols]
    asks = df[ask_cols]

    spread = asks - bids
    bad = asks.to_numpy() < bids.to_numpy()
    bad = pd.DataFrame(bad, index=df.index, columns=asks.columns)
    mask = bad.any(axis=1)

    rows = []

    for ts in df.index[mask]:
        bid_row = bids.loc[ts]
        ask_row = asks.loc[ts]
        bad_row = bad.loc[ts]

        for i in range(len(levels)):
            if bad_row.iloc[i]:
                rows.append(
                    {
                        "timestamp": ts,
                        "Level": levels[i],
                        "Bid": bid_row.iloc[i],
                        "Ask": ask_row.iloc[i],
                        "Spread": ask_row.iloc[i] - bid_row.iloc[i],
                    }
                )

    violations_df = pd.DataFrame(rows)

    print(f"\nProblematic timestamps ({len(violations_df)}) for spread in {filename}:")

    if not violations_df.empty:
        violations_df = violations_df.set_index("timestamp")
        display(violations_df)
    else:
        print("No violations found.")

    # add_issue(
    #     filename=filename, # type: ignore
    #     lob_type="FUT",
    #     id=key,
    #     description=f"{len(violations_df)} spread violation(s)."
    # )

In [ ]:
levels = range(1, config.LEVELS + 1)
bid_cols = [f"L{i}-BidPrice" for i in levels]
ask_cols = [f"L{i}-AskPrice" for i in levels]

idx = ctd_df.loc[~ctd_df["Overall spread check"]].index

display(ctd_df.loc[idx])

for id in idx:
    filename = ctd_df.loc[id, "Filename"]

    date = filename_to_date(filename)  # type: ignore
    year = str(date.year)  # type: ignore
    month = str(date.month).zfill(2)  # type: ignore

    df = pd.read_parquet(LOB_CTD_DIR / year / month / filename)  # type: ignore

    bids = df[bid_cols]
    asks = df[ask_cols]

    spread = asks - bids
    # bad = spread < 0
    # bad = asks.to_numpy() < bids.to_numpy()
    bad = asks.to_numpy() < bids.to_numpy()
    bad = pd.DataFrame(bad, index=df.index, columns=asks.columns)
    mask = bad.any(axis=1)

    rows = []

    for ts in df.index[mask]:
        bid_row = bids.loc[ts]
        ask_row = asks.loc[ts]
        bad_row = bad.loc[ts]

        for i in range(len(levels)):
            if bad_row.iloc[i]:
                rows.append(
                    {
                        "timestamp": ts,
                        "Level": levels[i],
                        "Bid": bid_row.iloc[i],
                        "Ask": ask_row.iloc[i],
                        "Spread": ask_row.iloc[i] - bid_row.iloc[i],
                    }
                )

    violations_df = pd.DataFrame(rows)

    print(f"\nProblematic timestamps ({len(violations_df)}) for spread in {filename}:")

    if not violations_df.empty:
        violations_df = violations_df.set_index("timestamp")
        display(violations_df)
    else:
        print("No violations found.")

    # add_issue(
    #     filename=filename, # type: ignore
    #     lob_type="CTD",
    #     id=key,
    #     description=f"{len(violations_df)} spread violation(s)."
    # )

## Volumes consistency

Check for volumes sign consistency in the dataset.

In [ ]:
key = "Volume sign"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_volume_sign.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures volumes consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_volume_sign.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd volumes consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "volume_sign.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
# fut_df = pd.read_parquet(OUT_INT_DIR / "fut_volume_sign.parquet")
# ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_volume_sign.parquet")

fut_summary = pd.DataFrame(
    {
        "File processed": len(fut_df),
        "LOB-wise bid volume violations": (~fut_df["Overall bid volume check"]).sum(),
        "LOB-wise bid volume violations (%)": (~fut_df["Overall bid volume check"]).mean() * 100,
        "LOB-wise ask volume violations": (~fut_df["Overall ask volume check"]).sum(),
        "LOB-wise ask volume violations (%)": (~fut_df["Overall ask volume check"]).mean() * 100,
        "Global bid consistency": fut_df["Overall bid volume check"].all(),
        "Global ask consistency": fut_df["Overall ask volume check"].all(),
        "Global overall consistency": fut_df["Overall volume check"].all(),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "File processed": len(ctd_df),
        "LOB-wise bid volume violations": (~ctd_df["Overall bid volume check"]).sum(),
        "LOB-wise bid volume violations (%)": (~ctd_df["Overall bid volume check"]).mean() * 100,
        "LOB-wise ask volume violations": (~ctd_df["Overall ask volume check"]).sum(),
        "LOB-wise ask volume violations (%)": (~ctd_df["Overall ask volume check"]).mean() * 100,
        "Global bid consistency": ctd_df["Overall bid volume check"].all(),
        "Global ask consistency": ctd_df["Overall ask volume check"].all(),
        "Global overall consistency": ctd_df["Overall volume check"].all(),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} volume consistency summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "Fut end time": time(23, 59, 59).strftime("%H:%M:%S"),
    "CTD start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "CTD end time": time(23, 59, 59).strftime("%H:%M:%S"),
}
display(summary)
display(summary.attrs)

# filename = "volume_consistency_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## Price consistency

Check for price ordering consistency in the dataset.

In [ ]:
key = "Price order"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_price_order.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures price order consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_price_order.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd price order consistency",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "price_order.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
fut_summary = pd.DataFrame(
    {
        "File processed": len(fut_df),
        "LOB-wise bid order violations": (~fut_df["Bid price decreasing ordering check"]).sum(),
        "LOB-wise bid order violations (%)": 100
        - fut_df["Bid price decreasing ordering check"].mean() * 100,
        "LOB-wise ask order violations": (~fut_df["Ask price increasing ordering check"]).sum(),
        "LOB-wise ask order violations (%)": 100
        - fut_df["Ask price increasing ordering check"].mean() * 100,
        "LOB-wise overall order violations": (~fut_df["Overall price order check"]).sum(),
        "LOB-wise overall order violations (%)": 100
        - fut_df["Overall price order check"].mean() * 100,
        "Global bid order": fut_df["Bid price decreasing ordering check"].all(),
        "Global ask order": fut_df["Ask price increasing ordering check"].all(),
        "Global overall order": fut_df["Overall price order check"].all(),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "File processed": len(ctd_df),
        "LOB-wise bid order violations": (~ctd_df["Bid price decreasing ordering check"]).sum(),
        "LOB-wise bid order violations (%)": 100
        - ctd_df["Bid price decreasing ordering check"].mean() * 100,
        "LOB-wise ask order violations": (~ctd_df["Ask price increasing ordering check"]).sum(),
        "LOB-wise ask order violations (%)": 100
        - ctd_df["Ask price increasing ordering check"].mean() * 100,
        "LOB-wise overall order violations": (~ctd_df["Overall price order check"]).sum(),
        "LOB-wise overall order violations (%)": 100
        - ctd_df["Overall price order check"].mean() * 100,
        "Global bid order": ctd_df["Bid price decreasing ordering check"].all(),
        "Global ask order": ctd_df["Ask price increasing ordering check"].all(),
        "Global overall order": ctd_df["Overall price order check"].all(),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} spread consistency summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "Fut end time": time(23, 59, 59).strftime("%H:%M:%S"),
    "CTD start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "CTD end time": time(23, 59, 59).strftime("%H:%M:%S"),
}
display(summary)
display(summary.attrs)

filename = "price_order_summary.parquet"
summary.to_parquet(OUT_PRO_DIR / filename)

In [ ]:
# Analyze failed LOBs bid
out_df = fut_df.copy()
idx = out_df.loc[~out_df["Bid price decreasing ordering check"], "Filename"].index
print(f"LOBs with bid price order issues ({len(idx)}):")
display(out_df.loc[idx])

for id in idx:
    filename = out_df.loc[id, "Filename"]
    date = filename_to_date(filename)
    year = str(date.year)  # type: ignore
    month = str(date.month).zfill(2)  # type: ignore

    df = pd.read_parquet(LOB_FUT_DIR / year / month / filename)

    bid_prices = df[[f"L{i}-BidPrice" for i in range(1, 11)]]

    diff = bid_prices.diff(axis=1)
    bad = diff > 0
    mask = bad.any(axis=1)

    rows = []
    cols = bid_prices.columns

    for ts in bid_prices.index[mask]:
        row = bid_prices.loc[ts]
        bad_row = bad.loc[ts]

        for j in range(1, len(cols)):
            if bad_row.iloc[j]:
                rows.append(
                    {
                        "timestamp": ts,
                        "Upper level": cols[j - 1],
                        "Upper level price": row.iloc[j - 1],
                        "Lower level": cols[j],
                        "Lower level price": row.iloc[j],
                        "Diff": row.iloc[j - 1] - row.iloc[j],
                    }
                )

    violations_df = pd.DataFrame(rows)
    violations_df.set_index("timestamp", inplace=True)

    print(f"\nProblematic timestamps ({len(violations_df)}) for bid order in {filename}:")

    if not violations_df.empty:
        display(violations_df)
    else:
        print("No violations found.")

    # add_issue(
    #     filename=filename,
    #     lob_type="FUT",
    #     id=key,
    #     description=f"{len(violations_df)} bid price order violation(s)"
    # )

In [ ]:
# Analyze failed LOBs ask
out_df = fut_df.copy()
idx = out_df.loc[~out_df["Ask price increasing ordering check"], "Filename"].index
print(f"LOBs with ask price order issues ({len(idx)}):")
display(out_df.loc[idx])

for id in idx:
    filename = out_df.loc[id, "Filename"]
    date = filename_to_date(filename)
    year = str(date.year)  # type: ignore
    month = str(date.month).zfill(2)  # type: ignore

    df = pd.read_parquet(LOB_FUT_DIR / year / month / filename)

    ask_prices = df[[f"L{i}-AskPrice" for i in range(1, 11)]]

    diff = ask_prices.diff(axis=1)
    bad = diff < 0
    mask = bad.any(axis=1)

    rows = []
    cols = ask_prices.columns

    for ts in ask_prices.index[mask]:
        row = ask_prices.loc[ts]
        bad_row = bad.loc[ts]

        for j in range(1, len(cols)):
            if bad_row.iloc[j]:
                rows.append(
                    {
                        "timestamp": ts,
                        "Upper level": cols[j - 1],
                        "Upper level price": row.iloc[j - 1],
                        "Lower level": cols[j],
                        "Lower level price": row.iloc[j],
                        "Diff": row.iloc[j - 1] - row.iloc[j],
                    }
                )

    violations_df = pd.DataFrame(rows)
    violations_df.set_index("timestamp", inplace=True)

    print(f"\nProblematic timestamps ({len(violations_df)}) for ask order in {filename}:")

    if not violations_df.empty:
        display(violations_df)
    else:
        print("No violations found.")

    # add_issue(
    #     filename = filename,
    #     lob_type = "FUT",
    #     id = key,
    #     description = f"{len(violations_df)} ask price order violation(s)"
    # )

## Timestamps Analysis

Analyze relevant timestastamps in the datasets.

In [ ]:
key = "Timestamps"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(0, 0, 0),
    end_time=time(23, 59, 59),
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(0, 0, 0),
    end_time=time(23, 59, 59),
)

In [ ]:
# fut_df = fut_results[key]
fut_df = pd.read_parquet(OUT_INT_DIR / "fut_timestamps.parquet")
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures timestamps analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

# ctd_df = ctd_results[key]
ctd_df = pd.read_parquet(OUT_INT_DIR / "ctd_timestamps.parquet")
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd timestamps analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "timestamps.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
fut_summary = pd.DataFrame(
    {
        "Min opening time": fut_df["First timestamp"].min(),
        "Min opening time (non-empty)": fut_df["First non-empty timestamp"].min(),
        "Max opening time": fut_df["First timestamp"].max(),
        "Max opening time (non-empty)": fut_df["First non-empty timestamp"].max(),
        "Min closing time": min(fut_df["Last timestamp"], key=time_to_seconds),
        "Min closing time (non-empty)": min(
            fut_df["Last non-empty timestamp"], key=time_to_seconds
        ),
        "Max closing time": max(fut_df["Last timestamp"], key=time_to_seconds),
        "Max closing time (non-empty)": max(
            fut_df["Last non-empty timestamp"], key=time_to_seconds
        ),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "Min opening time": ctd_df["First timestamp"].min(),
        "Min opening time (non-empty)": ctd_df["First non-empty timestamp"].min(),
        "Max opening time": ctd_df["First timestamp"].max(),
        "Max opening time (non-empty)": ctd_df["First non-empty timestamp"].max(),
        "Min closing time": ctd_df["Last timestamp"].min(),
        "Min closing time (non-empty)": ctd_df["Last non-empty timestamp"].min(),
        "Max closing time": ctd_df["Last timestamp"].max(),
        "Max closing time (non-empty)": ctd_df["Last non-empty timestamp"].max(),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} relevant timestamps summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": "00:00:00",
    "Fut end time": "23:59:59",
    "CTD start time": "00:00:00",
    "CTD end time": "23:59:59",
}
display(summary)
display(summary.attrs)

# filename = "relevant_timestamps_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

In [ ]:
# Inspect anomalous timestamps
for col in [
    "First timestamp",
    "First non-empty timestamp",
    "Last timestamp",
    "Last non-empty timestamp",
]:
    display(
        pd.DataFrame(
            fut_df[col].unique(),
            columns=[col],
        )
        .set_index(col)
        .sort_index()
    )


mask = fut_df["Last non-empty timestamp"] == time(14, 36, 40)
row = fut_df.loc[mask]
display(row)

### Plots

In [ ]:
# Configuration for timestamp distribution plot
lob_type = "ctd"
timestamp_col = "First timestamp"

# Load data
filename = f"{lob_type}_timestamps.parquet"
df = pd.read_parquet(OUT_INT_DIR / filename)

# Create distribution plot
fig = plot_timestamp_distribution(
    df=df,
    timestamp_col=timestamp_col,
    bin_start_hh_mm="07:30",
    bin_width_seconds=60,
    major_tick_minutes=10,
)

plt.show()

# figure_name = f"{lob_type}_hist_{timestamp_col.lower().replace(' ', '_')}"
# fig.savefig(OUT_FIG_DIR / f"{figure_name}.png", dpi=300, bbox_inches="tight")